Good. Now we move to **Parts 14–15: interpreting XGBoost with PDP and SHAP**. This is especially useful because you’ve already started learning PDP, so now you’ll see how it applies to a powerful boosting model.

# XGBoost — PDP & SHAP for Model Interpretation

---

# Part 14 — Partial Dependence Plot (PDP)

## 1. Why do we need PDP?

XGBoost can give excellent predictions, but it is often difficult to understand **why** the model makes those predictions.

For example, suppose your XGBoost model predicts house prices.

You may want to know:

> "How does `MedInc` affect the predicted house price?"

A simple feature importance plot can tell you:

```text
MedInc → very important
```

But it does **not** tell you:

> "As MedInc increases, does the predicted house price increase or decrease?"

That's where PDP comes in.

---

# 2. What is Partial Dependence?

**Partial Dependence Plot (PDP)** shows the average effect of one or more features on the model's prediction.

For one feature \(x_j\):

$$
PD(x_j)
=
\frac{1}{n}
\sum_{i=1}^{n}
f(x_j,x_{i,-j})
$$

where:

* \(x_j\) = feature we're interested in
* \(x_{i,-j}\) = all other features for observation \(i\)
* \(f\) = trained model
* \(n\) = number of observations

In simpler words:

> PDP changes one feature while keeping the other features from the dataset, then averages the model's predictions.

---

# 3. Simple Example

Suppose our model has:

```text
MedInc
HouseAge
AveRooms
Population
```

We want to understand:

```text
MedInc → House Price
```

PDP might show:

```text
Predicted
House Price
   ↑
   │                  ______
   │              ___/
   │          ___/
   │      ___/
   │_____/
   └──────────────────────→
             MedInc
```

This tells us that the model generally predicts higher house prices as `MedInc` increases.

---

# 4. PDP Doesn't Mean Causation

This is extremely important.

If PDP shows:

$$
MedInc \uparrow
\Rightarrow
PredictedPrice \uparrow
$$

you should **not** conclude:

> "Increasing someone's income causes the house price to increase."

PDP describes the **model's learned relationship**, not a causal relationship.

Remember:

$$
\boxed{
Model\ relationship \neq Causal\ relationship
}
$$

---

# 5. Using PDP with XGBoost

Scikit-learn provides:

```python
from sklearn.inspection import PartialDependenceDisplay
```

Suppose you've trained:

```python
model
```

Then:

```python
from sklearn.inspection import PartialDependenceDisplay
import matplotlib.pyplot as plt

PartialDependenceDisplay.from_estimator(
    model,
    X_train,
    ["MedInc"]
)

plt.show()
```

This produces the PDP for `MedInc`.

---

# 6. Multiple Features

You can inspect multiple features:

```python
features = [
    "MedInc",
    "HouseAge",
    "AveRooms"
]

PartialDependenceDisplay.from_estimator(
    model,
    X_train,
    features
)

plt.show()
```

This allows you to understand several features at once.

---

# 7. Two-Feature PDP

PDP can also examine interactions between two features.

For example:

```python
PartialDependenceDisplay.from_estimator(
    model,
    X_train,
    [("MedInc", "HouseAge")]
)

plt.show()
```

Now you're asking:

> "How does the model's prediction change depending on both income and house age?"

This can reveal interactions that aren't visible in a one-feature PDP.

---

# 8. PDP for XGBoost — Complete Example

Using your California Housing experiment:

```python
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from sklearn.inspection import PartialDependenceDisplay

import matplotlib.pyplot as plt

data = fetch_california_housing(as_frame=True)

X = data.data
y = data.target

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    random_state=42
)

model.fit(X_train, y_train)

PartialDependenceDisplay.from_estimator(
    model,
    X_train,
    ["MedInc"]
)

plt.show()
```

---

# 9. PDP Interpretation

Suppose the PDP looks roughly like:

```text
Prediction
   ↑
   │                  ______
   │              ___/
   │           __/
   │       ___/
   │_____/
   └────────────────────→
             MedInc
```

Interpretation:

```text
Low MedInc
     ↓
Lower predicted house value

MedInc increases
     ↓
Predicted value increases

Very high MedInc
     ↓
Effect starts becoming smaller
```

The flattening section indicates **diminishing marginal effect according to the model**.

---

# 10. PDP With Classification

For classification, PDP can show the effect on a model's predicted probability.

Example:

```python
PartialDependenceDisplay.from_estimator(
    classifier,
    X_train,
    ["age"]
)

plt.show()
```

Depending on the estimator and settings, the plot represents the model's response/probability for the relevant class.

---

# 11. Important PDP Limitation

PDP assumes that the feature being investigated can be varied somewhat independently of the other features.

This can become problematic when features are highly correlated.

For example:

```text
House size
Number of rooms
Number of bedrooms
```

are naturally related.

PDP might evaluate combinations such as:

```text
Huge house
+
Very few rooms
```

that rarely occur in reality.

Therefore:

$$
\boxed{
Correlated\ features
\Rightarrow
PDP\ interpretation\ can\ become\ unreliable
}
$$

---

# Part 15 — SHAP

Now we move to one of the most important modern model-interpretation techniques.

# 12. What is SHAP?

**SHAP = SHapley Additive exPlanations**

SHAP is based on **Shapley values** from cooperative game theory.

The basic question is:

> **How much did each feature contribute to this particular prediction?**

This is different from ordinary feature importance.

---

# 13. Feature Importance vs PDP vs SHAP

Think about these three techniques:

### Feature Importance

Answers:

> "Which features are generally important to the model?"

```text
MedInc       ██████████
AveRooms     ███████
HouseAge     █████
Population   ███
```

---

### PDP

Answers:

> "How does the model's prediction generally change as this feature changes?"

```text
MedInc ↑
   ↓
Prediction ↑
```

---

### SHAP

Answers:

> "For this specific prediction, how much did each feature contribute?"

```text
Prediction = 4.2

MedInc      +1.2
AveRooms    +0.4
HouseAge    -0.2
Population  -0.1
```

This is the key distinction.

---

# 14. SHAP Intuition

Imagine the model predicts:

$$
Prediction = 4.2
$$

Suppose the average prediction is:

$$
Base = 2.1
$$

Then the model moved from:

$$
2.1
$$

to:

$$
4.2
$$

because of feature contributions.

For example:

$$
4.2
=
2.1
+
1.4
+
0.5
+
0.3
-
0.1
$$

So:

```text
Base prediction       2.1
MedInc contribution   +1.4
AveRooms contribution +0.5
HouseAge contribution +0.3
Population            -0.1
                       ───
Final prediction       4.2
```

That's the central idea of SHAP.

---

# 15. SHAP Mathematical Idea

SHAP assigns each feature a Shapley value:

$$
\phi_j
$$

The prediction can be represented as:

$$
\boxed{
f(x)
=
\phi_0
+
\sum_{j=1}^{M}\phi_j
}
$$

where:

* \(\phi_0\) = base value
* \(\phi_j\) = contribution of feature \(j\)
* \(M\) = number of features

So:

$$
\boxed{
Prediction = Base\ Value + Feature\ Contributions
}
$$

---

# 16. Installing SHAP

Install:

```bash
pip install shap
```

Then:

```python
import shap
```

---

# 17. SHAP with XGBoost

Suppose you've trained:

```python
model = XGBRegressor(...)
```

Create an explainer:

```python
explainer = shap.TreeExplainer(model)
```

Then calculate SHAP values:

```python
shap_values = explainer.shap_values(X_test)
```

For modern SHAP versions, you can also use:

```python
explainer = shap.Explainer(model, X_train)

shap_values = explainer(X_test)
```

The latter is generally the cleaner modern API.

---

# 18. SHAP Summary Plot

One of the most useful SHAP plots:

```python
shap.summary_plot(
    shap_values,
    X_test
)
```

This tells you:

* Feature importance
* Direction of influence
* Distribution of SHAP values

---

# 19. Understanding a SHAP Summary Plot

Conceptually:

```text
                 SHAP value
                     →
MedInc       • • • • • • • •
AveRooms       • • • • • •
HouseAge        • • • •
Population       • • •
```

The horizontal position represents the magnitude and direction of the contribution.

Generally:

```text
Left of 0  → pushes prediction down
Right of 0 → pushes prediction up
```

The color commonly represents feature value:

```text
High feature value
Low feature value
```

So you can answer both:

> How important is this feature?

and:

> Does a high/low value tend to push the prediction up or down?

---

# 20. SHAP Bar Plot

You can also create a simpler global importance plot:

```python
shap.summary_plot(
    shap_values,
    X_test,
    plot_type="bar"
)
```

This gives something conceptually like:

```text
MedInc       ███████████
AveRooms     ████████
HouseAge     █████
AveOccup     ████
Population   ███
```

This is useful for comparing overall feature importance.

---

# 21. SHAP Waterfall Plot

Now we move from global interpretation to **individual predictions**.

Suppose we want to understand prediction number 0.

```python
shap.plots.waterfall(
    shap_values[0]
)
```

This shows how each feature moved the prediction away from the base value.

Conceptually:

```text
Base value
   │
   ├── MedInc      +1.2
   │
   ├── AveRooms    +0.4
   │
   ├── HouseAge   -0.2
   │
   └── Population -0.1
   │
   ↓
Final prediction
```

This is extremely useful when explaining **one specific prediction**.

---

# 22. SHAP Dependence Plot

You can also investigate how a particular feature affects predictions:

```python
shap.dependence_plot(
    "MedInc",
    shap_values.values,
    X_test
)
```

Depending on the SHAP API/version, the exact object passed may differ.

The basic idea is:

```text
Feature value
      ↓
SHAP contribution
```

This is somewhat related to PDP but gives more detailed observation-level information.

---

# 23. PDP vs SHAP

This is extremely important.

| PDP                                   | SHAP                                 |
| ------------------------------------- | ------------------------------------ |
| Global interpretation                 | Global + local                       |
| Average effect                        | Individual contributions             |
| Shows relationship                    | Shows contribution                   |
| Can struggle with correlated features | Also has correlation caveats         |
| Easier to understand                  | More detailed                        |
| Model-agnostic interface              | Especially efficient for tree models |

---

# 24. Example

Suppose XGBoost predicts:

$$
HousePrice=4.5
$$

### PDP

might tell us:

> Generally, higher `MedInc` increases predicted house value.

### SHAP

for one house might tell us:

```text
Base prediction       2.0
MedInc               +1.7
AveRooms             +0.5
HouseAge             +0.2
Population           -0.1
Other features       +0.2
                     ────
Prediction            4.5
```

So PDP gives the **general relationship**, while SHAP can explain **this particular prediction**.

---

# 25. Complete XGBoost + SHAP Example

```python
import shap
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

from xgboost import XGBRegressor


# -------------------------
# Load data
# -------------------------

data = fetch_california_housing(as_frame=True)

X = data.data
y = data.target


# -------------------------
# Split
# -------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


# -------------------------
# XGBoost
# -------------------------

model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(X_train, y_train)


# -------------------------
# SHAP Explainer
# -------------------------

explainer = shap.Explainer(
    model,
    X_train
)

shap_values = explainer(X_test)


# -------------------------
# Summary Plot
# -------------------------

shap.summary_plot(
    shap_values,
    X_test
)
```

---

# 26. SHAP Bar Plot

```python
shap.summary_plot(
    shap_values,
    X_test,
    plot_type="bar"
)
```

This provides a global ranking of mean absolute SHAP contribution.

---

# 27. Explain One Observation

```python
shap.plots.waterfall(
    shap_values[0]
)
```

Now you can explain:

> Why did XGBoost make this particular prediction?

This is called **local interpretability**.

---

# 28. Global vs Local Interpretability

You should remember this distinction.

### Global

Understanding the model as a whole.

Examples:

```text
Feature importance
PDP
SHAP summary plot
```

Question:

> "How does the model generally behave?"

---

### Local

Understanding one prediction.

Examples:

```text
SHAP waterfall
SHAP force-style explanation
```

Question:

> "Why did the model make this specific prediction?"

---

# 29. Why SHAP Is Powerful for XGBoost

Tree-based models can have complex nonlinear relationships.

For example:

```text
MedInc
   ↓
House Price
   ↓
nonlinear relationship
```

and interactions:

```text
MedInc + HouseAge
        ↓
House Price
```

SHAP can help expose these relationships at both the global and local level.

---

# 30. The Complete Interpretation Stack

For your XGBoost project, use the tools in this order:

```text
                 XGBoost Model
                       │
          ┌────────────┼────────────┐
          ↓            ↓            ↓
     Feature        PDP          SHAP
    Importance        │             │
          │           │       ┌─────┴─────┐
          │           │       ↓           ↓
          │           │    Global       Local
          │           │    SHAP         SHAP
          │           │
          ↓           ↓
     Which features   How feature
     matter?          affects prediction?
```

---

# 31. Which One Should You Use?

### If your question is:

**"Which features are important?"**

Use:

```text
Feature Importance
SHAP Bar Plot
```

### If your question is:

**"How does feature X affect predictions?"**

Use:

```text
PDP
SHAP Dependence
```

### If your question is:

**"Why did the model make this particular prediction?"**

Use:

```text
SHAP Waterfall
```

---

# 32. Important Warning: Interpretation ≠ Causation

This applies to both PDP and SHAP.

Suppose SHAP tells you:

```text
MedInc → +1.5
```

This means:

> For this model prediction, `MedInc` contributed +1.5 relative to the model's baseline.

It does **not** mean:

> Increasing MedInc by one unit will causally increase the house price by 1.5.

That's a crucial distinction.

---

# 33. Your XGBoost Learning Progression

You've now covered:

```text
Part 1
XGBoost Introduction
        ↓
Part 2
Gradient Boosting → XGBoost
        ↓
Part 3
XGBoost Mathematics
        ↓
Part 4
Gradient + Hessian
        ↓
Part 5
Tree Construction + Gain
        ↓
Part 6
Regularization
        ↓
Part 7
Hyperparameters
        ↓
Part 8
XGBClassifier
        ↓
Part 9
XGBRegressor
        ↓
Part 10
Evaluation
        ↓
Part 11
Hyperparameter Tuning
        ↓
Part 12
Early Stopping
        ↓
Part 13
Feature Importance
        ↓
Part 14
PDP
        ↓
Part 15
SHAP
```

So you've essentially completed the **core XGBoost theory + practical implementation + interpretability** portion.

## Next: Part 16 — Complete XGBoost Project

The best next exercise for you is to take the **California Housing dataset you've already used** and build a proper comparison:

```text
DecisionTreeRegressor
        ↓
RandomForestRegressor
        ↓
AdaBoostRegressor
        ↓
GradientBoostingRegressor
        ↓
XGBRegressor
```

Then compare:

```text
R²
MSE
RMSE
Training time
Overfitting gap
Feature importance
PDP
SHAP
```

That will tie together almost everything you've learned in **Decision Trees → Ensemble Learning → AdaBoost → Gradient Boosting → XGBoost**.
